In [7]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "twcs"

SAMPLE_PATH = RAW_DATA_DIR / "sample.csv"
FULL_DATA_PATH = RAW_DATA_DIR / "twcs.csv"

sample_df = pd.read_csv(SAMPLE_PATH)

print(f"Sample shape: {sample_df.shape}")
print(f"Columns: {sample_df.columns.tolist()}")

Sample shape: (93, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [8]:
sample_df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,119237,105834,True,Wed Oct 11 06:55:44 +0000 2017,@AppleSupport causing the reply to be disregar...,119236,NaN
1,119238,ChaseSupport,False,Wed Oct 11 13:25:49 +0000 2017,@105835 Your business means a lot to us. Pleas...,NaN,119239.0
2,119239,105835,True,Wed Oct 11 13:00:09 +0000 2017,@76328 I really hope you all change but I'm su...,119238,NaN
3,119240,VirginTrains,False,Tue Oct 10 15:16:08 +0000 2017,@105836 LiveChat is online at the moment - htt...,119241,119242.0
4,119241,105836,True,Tue Oct 10 15:17:21 +0000 2017,@VirginTrains see attached error message. I've...,119243,119240.0


In [9]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   tweet_id                 93 non-null     int64  
 1   author_id                93 non-null     object 
 2   inbound                  93 non-null     bool   
 3   created_at               93 non-null     object 
 4   text                     93 non-null     object 
 5   response_tweet_id        65 non-null     object 
 6   in_response_to_tweet_id  68 non-null     float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 4.6+ KB


In [10]:
sample_df.isna().sum()

tweet_id                    0
author_id                   0
inbound                     0
created_at                  0
text                        0
response_tweet_id          28
in_response_to_tweet_id    25
dtype: int64

In [11]:
sample_df["inbound"].value_counts()

inbound
True     49
False    44
Name: count, dtype: int64

In [12]:
full_dataset_size_gb = FULL_DATA_PATH.stat().st_size / (1024 ** 3)

print(f"Full dataset size: {full_dataset_size_gb:.2f} GB")

Full dataset size: 0.48 GB


In [13]:
df = pd.read_csv(FULL_DATA_PATH)

print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB")

Shape: (2811774, 7)
Memory usage: 1076.56 MB


In [14]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB


In [16]:
brand_counts = (
    df.loc[df["inbound"] == False, "author_id"]
      .value_counts()
)

print(f"Number of support accounts: {len(brand_counts)}")
display(brand_counts.head(30))

Number of support accounts: 108


author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
VerizonSupport      17966
UPSHelp             17817
ATVIAssist          17650
O2                  16212
Safaricom_Care      16077
idea_cares          15724
AskTarget           13218
AirAsiaSupport      12829
BofA_Help           12683
SW_Help             12231
Name: count, dtype: int64

In [17]:
candidate_brands = [
    "AppleSupport",
    "AmazonHelp",
    "Uber_Support",
    "SpotifyCares",
    "Delta"
]

candidate_df = df[
    (df["author_id"].isin(candidate_brands)) &
    (df["inbound"] == False)
].copy()

brand_analysis = (
    candidate_df
    .groupby("author_id")
    .agg(
        total_brand_tweets=("tweet_id", "count"),
        replies_to_customer=("in_response_to_tweet_id", "count"),
        unique_customers=("in_response_to_tweet_id", "nunique")
    )
)

brand_analysis["reply_rate"] = (
    brand_analysis["replies_to_customer"] /
    brand_analysis["total_brand_tweets"]
)

brand_analysis.sort_values(
    "total_brand_tweets",
    ascending=False
)

,total_brand_tweets,replies_to_customer,unique_customers,reply_rate
author_id,,,,
AmazonHelp,169840,169287,155445,0.996744
AppleSupport,106860,106719,106696,0.998681
Uber_Support,56270,56261,55283,0.999840
SpotifyCares,43265,43243,41734,0.999492
Delta,42253,42197,36215,0.998675


In [18]:
# Build a lookup from tweet_id -> tweet text
tweet_text = df.set_index("tweet_id")["text"]

# Get brand replies from our five candidates
brand_replies = df[
    (df["author_id"].isin(candidate_brands)) &
    (df["inbound"] == False) &
    (df["in_response_to_tweet_id"].notna())
].copy()

# Find the customer tweet that each brand reply is responding to
brand_replies["customer_message"] = (
    brand_replies["in_response_to_tweet_id"]
    .map(tweet_text)
)

conversation_pairs = brand_replies[
    brand_replies["customer_message"].notna()
][[
    "author_id",
    "customer_message",
    "text"
]].rename(columns={
    "author_id": "brand",
    "text": "brand_response"
})

print(f"Total customer → brand pairs: {len(conversation_pairs):,}")

display(
    conversation_pairs
    .groupby("brand")
    .size()
    .sort_values(ascending=False)
)

Total customer → brand pairs: 417,019


brand
AmazonHelp      168823
AppleSupport    106648
Uber_Support     56193
SpotifyCares     43206
Delta            42149
dtype: int64

In [19]:
for brand in candidate_brands:
    print(f"\n{'=' * 70}")
    print(f"{brand}")
    print(f"{'=' * 70}")

    display(
        conversation_pairs[
            conversation_pairs["brand"] == brand
        ]
        .sample(5, random_state=42)
        [["customer_message", "brand_response"]]
    )


AppleSupport


,customer_message,brand_response
1551498,@AppleSupport I️ can connect just fine but my ...,@516039 Understood. Check here under step 6: ...
2273680,So @AppleSupport my speaker just gon stop work...,@273855 We'd like to help. Tell us what device...
194358,"@AppleSupport look, i just got this iPhone 8 a...",@170489 We want to make sure your iPhone is th...
2773263,@814571 @AppleSupport please update the publis...,@814570 We are seeing todays date at this link...
253910,@AppleSupport recent update for IPad and IPhon...,@163247 Let us know which apps you're having t...



AmazonHelp


,customer_message,brand_response
1744101,Yo necesitaba elevar el espejo de mi tocador y...,"@565800 Hola, Aranzazu. ¡Una solución intelige..."
1866765,@AmazonHelp Once You Receive..? I did't Send b...,@367525 Sorry for the confusion. Request you t...
1503584,@AmazonHelp so I’ve had an issue and no one ha...,@479947 We'd love to take a closer look into y...
940152,@AmazonHelp Thank you,"@366678 You are welcome, Akshay. ^AU"
392991,@137605 Ayer no me entregaron un paquete de @1...,"@221198 Hola, lamentamos los inconvenientes qu..."



Uber_Support


,customer_message,brand_response
2182291,@Uber_Support I am overcharged by UBER without...,"@677936 Here to help! Send us a note here, htt..."
626961,@115873 how do I get in contact with you due t...,@287971 Here to help! Send us a note via https...
2258795,Uninstalling y'all suck @115873 @Uber_Support,@695893 Here to help! Can you please send us a...
2035686,@Uber_Support I got an e-mail that I need to u...,@642075 We're here to help. Please DM us the e...
332563,Can @115873 end car ownership?,@206243 Happy to help! Can you please provide ...



SpotifyCares


,customer_message,brand_response
190592,Can someone please tell me how to start an art...,"@169626 Hey there! At this time, we're afraid ..."
1922714,@SpotifyCares having trouble activating a Fami...,"@614298 Hey Matt, help's here! Can you DM us y..."
1701489,@554978 ただspotifyは無料会員だと、シャッフルでしか再生できなかったり途中で関...,@554977 ツイート拝見しました。Freeユーザーでスマホをご利用の場合はシャッフル再生...
2496510,"@SpotifyCares Thank you, I know that it’s some...",@751180 We've already passed your feedback on ...
2308144,"@SpotifyCares It's not song specific, but it m...",@707288 We understand. Could you send us a DM ...



Delta


,customer_message,brand_response
224858,@Delta So why not move me to one thats not bro...,"@178116 Anson, please check with the flight cr..."
1333298,@Delta offers 2500 miles for a late bag and 45...,@460737 ...future travel needs. Have a good ev...
842366,@delta why would promise a customer $150 of th...,@342421 Hi there. I'm sorry for any inconvenie...
458272,@Delta i have an international flight schedule...,@240008 Hi there. Please share your confirmati...
1587473,As always @Delta got me to my destination ahea...,@525068 That's awesome sauce! Thanks for enjo...


In [20]:
apple_df = conversation_pairs[
    conversation_pairs["brand"] == "AppleSupport"
].copy()

print(f"AppleSupport pairs: {len(apple_df):,}")

print("\nAverage customer message length:")
print(apple_df["customer_message"].str.len().mean())

print("\nAverage brand response length:")
print(apple_df["brand_response"].str.len().mean())

print("\nCustomer messages with very short text:")
print(
    (apple_df["customer_message"].str.len() < 20).mean() * 100,
    "%"
)

AppleSupport pairs: 106,648

Average customer message length:
109.26146763183557

Average brand response length:
136.62351849073588

Customer messages with very short text:
0.8138924311754556 %


In [21]:
display(
    apple_df[
        ["customer_message", "brand_response"]
    ].sample(50, random_state=42)
)

,customer_message,brand_response
1551498,@AppleSupport I️ can connect just fine but my ...,@516039 Understood. Check here under step 6: ...
2273680,So @AppleSupport my speaker just gon stop work...,@273855 We'd like to help. Tell us what device...
194358,"@AppleSupport look, i just got this iPhone 8 a...",@170489 We want to make sure your iPhone is th...
2773263,@814571 @AppleSupport please update the publis...,@814570 We are seeing todays date at this link...
253910,@AppleSupport recent update for IPad and IPhon...,@163247 Let us know which apps you're having t...
2601024,@AppleSupport I did the latest update at that ...,@774928 Please DM us the software version you’...
96449,@AppleSupport @115858 why isn’t my low power m...,@143274 Thanks for reaching out. We're here to...
2061261,@AppleSupport help me.. my iPhone 5s is freezi...,@648515 We'd love to help. Could you Direct Me...
404268,Another @115858 #ios11 bug - search textbox in...,@224486 Let’s investigate this together. Use t...
1394221,@AppleSupport ETA on iPhone activation service...,@476002 Follow the steps listed her to activat...


In [22]:
# Inspect how many AppleSupport conversations have multiple customer messages
# connected to the same support interaction.

apple_replies = df[
    (df["author_id"] == "AppleSupport") &
    (df["inbound"] == False) &
    (df["in_response_to_tweet_id"].notna())
].copy()

# Count how many AppleSupport replies point to each customer tweet
reply_counts = (
    apple_replies["in_response_to_tweet_id"]
    .value_counts()
    .rename("apple_response_count")
)

# Count how many times each customer tweet is itself replied to by AppleSupport
print("AppleSupport reply tweets:", len(apple_replies))
print("\nTweets receiving multiple AppleSupport replies:")
print(reply_counts[reply_counts > 1].head(20))

print("\nDistribution of AppleSupport responses per customer tweet:")
print(reply_counts.value_counts().sort_index().head(10))

AppleSupport reply tweets: 106719

Tweets receiving multiple AppleSupport replies:
in_response_to_tweet_id
1586869.0    2
446503.0     2
1584584.0    2
1477619.0    2
236438.0     2
1687724.0    2
1537598.0    2
1624949.0    2
1531314.0    2
294915.0     2
1703866.0    2
1748004.0    2
1774859.0    2
1905789.0    2
2007522.0    2
2064278.0    2
1772889.0    2
2253541.0    2
2292433.0    2
2450944.0    2
Name: apple_response_count, dtype: int64

Distribution of AppleSupport responses per customer tweet:
apple_response_count
1    106673
2        23
Name: count, dtype: int64


In [24]:
# Find very short customer messages that may lack enough context
# to be useful as standalone support examples.

apple_df["customer_length"] = apple_df["customer_message"].str.len()

short_messages = (
    apple_df[apple_df["customer_length"] <= 40]
    .sort_values("customer_length")
    [["customer_message", "brand_response", "customer_length"]]
)

print("Customer messages <= 40 characters:", len(short_messages))
print(f"Percentage: {len(short_messages) / len(apple_df) * 100:.2f}%")

print("\nSample of short messages:")
display(
    short_messages.sample(
        min(50, len(short_messages)),
        random_state=42
    )
)

Customer messages <= 40 characters: 8134
Percentage: 7.63%

Sample of short messages:


,customer_message,brand_response,customer_length
1372948,WHY IM SEEING BOXES FIX THIS @115858,@470691 Thanks for reaching out. Could you tel...,36
225272,@AppleSupport I’m confused help me,@178226 We'd like to see how we can help. Can ...,34
1665858,@115858 @AppleSupport,@546035 We’re happy to help with any complicat...,21
631622,@AppleSupport iOS11.0.0 on an iPhone 6s,@289254 That is expected behavior. Apple encou...,39
1273494,@AppleSupport I need your help,@446436 We're here! How can we help you?,30
796141,@AppleSupport 11.0.1,@330819 We'd recommend backing up your iPhone ...,20
65024,@AppleSupport fix my phone.,@134144 We're happy to help. Could you please ...,27
375508,@AppleSupport Version 11.1,@216970 Thanks for that information. Let's con...,26
2512280,@AppleSupport Great. Thank you,@456048 You're most welcome!,30
2258367,@AppleSupport 7plus and home@wifi,@277899 Thanks for confirming. Let's move to D...,33


In [25]:
import re

# Patterns that usually indicate a follow-up, acknowledgement,
# or message without enough standalone information.
low_info_pattern = re.compile(
    r"^\s*("
    r"thanks?(\s+(you|a lot|so much))?"
    r"|thank you(\s+(so much|very much))?"
    r"|you'?re welcome"
    r"|yes"
    r"|yeah"
    r"|yep"
    r"|no"
    r"|nope"
    r"|ok"
    r"|okay"
    r"|alright"
    r"|sure"
    r"|same"
    r"|same for me"
    r"|me too"
    r"|got it"
    r"|that worked"
    r"|it worked"
    r"|works"
    r"|please check dm"
    r"|check dm"
    r"|check your dm"
    r"|dm me"
    r"|hello"
    r"|hi"
    r"|hey"
    r"|help"
    r"|i need your help"
    r"|@AppleSupport"
    r")\s*[.!?]*\s*$",
    re.IGNORECASE
)

apple_df["is_low_info"] = (
    apple_df["customer_message"]
    .fillna("")
    .str.strip()
    .apply(lambda x: bool(low_info_pattern.match(x)))
)

low_info_df = apple_df[apple_df["is_low_info"]].copy()

print("Obvious low-information messages:", len(low_info_df))
print(f"Percentage: {len(low_info_df) / len(apple_df) * 100:.2f}%")

print("\nExamples:")
display(
    low_info_df[
        ["customer_message", "brand_response", "customer_length"]
    ].sample(
        min(50, len(low_info_df)),
        random_state=42
    )
)

Obvious low-information messages: 125
Percentage: 0.12%

Examples:


,customer_message,brand_response,customer_length
500167,@AppleSupport,@252368 Thanks for bringing this to our attent...,13
1244892,@AppleSupport,@439832 Here's some info on that: https://t.co...,13
1056274,@AppleSupport,@394966 We're glad to help! Have you restarted...,13
1818054,@AppleSupport,@584546 Let's get you some assistance on this ...,13
1446759,@AppleSupport,@489157 We'd be happy to help out! Send us a D...,13
1871038,@AppleSupport,@599478 Here’s what you can do to work around ...,13
1496273,@AppleSupport,@502112 We offer support via Twitter in Englis...,13
2725184,@applesupport,@803444 We'd like to help. What model and soft...,13
1895565,@AppleSupport,@606521 We'd like to see how we can help with ...,13
87199,@AppleSupport,@140662 Hey there! We've received your DM and ...,13


In [27]:
# Get the AppleSupport reply records again from the existing df.
# We use tweet IDs to preserve the correct relationships.

apple_replies = df[
    (df["author_id"] == "AppleSupport") &
    (df["inbound"] == False) &
    (df["in_response_to_tweet_id"].notna())
].copy()

# The customer tweet that AppleSupport replied to
apple_replies["customer_tweet_id"] = (
    apple_replies["in_response_to_tweet_id"]
)

# Find what each customer tweet was replying to.
# This lets us detect multi-turn/context-dependent messages.
parent_lookup = (
    df.set_index("tweet_id")["in_response_to_tweet_id"]
)

apple_replies["parent_tweet_id"] = (
    apple_replies["customer_tweet_id"]
    .map(parent_lookup)
)

# Get the actual parent tweet text
text_lookup = df.set_index("tweet_id")["text"]

apple_replies["parent_text"] = (
    apple_replies["parent_tweet_id"]
    .map(text_lookup)
)

# Get who authored the parent tweet
author_lookup = df.set_index("tweet_id")["author_id"]

apple_replies["parent_author"] = (
    apple_replies["parent_tweet_id"]
    .map(author_lookup)
)

# Keep only rows where the customer message itself has a parent tweet
has_parent = apple_replies["parent_tweet_id"].notna()

print("Total AppleSupport customer interactions:", len(apple_replies))
print("Customer messages replying to another tweet:", has_parent.sum())
print(f"Percentage: {has_parent.mean() * 100:.2f}%")

print("\nWho wrote the parent tweet?")
print(
    apple_replies.loc[has_parent, "parent_author"]
    .value_counts()
    .head(10)
)

print("\nSample conversations with previous-tweet context:")
display(
    apple_replies.loc[
        has_parent,
        [
            "parent_text",
            "text",
            "customer_tweet_id"
        ]
    ]
    .rename(columns={
        "text": "customer_message"
    })
    .sample(
        min(50, has_parent.sum()),
        random_state=42
    )
)

Total AppleSupport customer interactions: 106719
Customer messages replying to another tweet: 32014
Percentage: 30.00%

Who wrote the parent tweet?
parent_author
AppleSupport    23388
115858            169
116333             83
342218             23
180336             13
135834             11
135589             11
123379             10
219683              9
681228              8
Name: count, dtype: int64

Sample conversations with previous-tweet context:


,parent_text,customer_message,customer_tweet_id
2300691,Hello @115858 my iPhone appears gaussian blurr...,@698609 We’re glad we could assist you. Please...,2434628.0
1072783,After updating iOS 11 this what happen to iPho...,@399021 We'd love to help figure this out with...,1186611.0
911557,@359543 Thanks. Have you tried restarting your...,@359543 Let's have you try the steps here: htt...,1012348.0
2378042,"In iOS 11, you've got more control over the Co...",@723532 We can help get you to the right place...,661578.0
1965482,@160683 Good question. A password is required ...,@160683 It will only remove content/photos sav...,2121600.0
1180944,@AppleSupport + if I can’t post a HD video on ...,@424773 We're glad you resolved this. We also ...,1304261.0
1349595,@465056 I️ don’t know what to do,@465056 We'd love to look into this further wi...,1485872.0
2184096,@AppleSupport Thank you! It's 11.1.1. But the ...,@678382 We can take a look at this with you. S...,2346069.0
1704319,@555637 Thanks for reaching out. Which iPhone ...,@555637 We're glad to hear that's working agai...,1858197.0
291048,@195274 We'd like to help. Let's see if backin...,@195274 Thank you for letting us know. So we c...,332847.0


In [28]:
# Separate contextual customer messages based on who wrote
# the previous tweet.

contextual = apple_replies[
    apple_replies["parent_tweet_id"].notna()
].copy()

customer_replying_to_apple = contextual[
    contextual["parent_author"] == "AppleSupport"
]

customer_replying_to_customer = contextual[
    contextual["parent_author"] != "AppleSupport"
]

print("Customer messages with previous context:", len(contextual))
print(
    "Replying to previous AppleSupport message:",
    len(customer_replying_to_apple)
)
print(
    "Replying to another user's message:",
    len(customer_replying_to_customer)
)

print("\n--- Customer replying to AppleSupport ---")
display(
    customer_replying_to_apple[
        ["parent_text", "text", "customer_tweet_id"]
    ]
    .rename(columns={"text": "customer_message"})
    .sample(
        min(25, len(customer_replying_to_apple)),
        random_state=42
    )
)

print("\n--- Customer replying to another user ---")
display(
    customer_replying_to_customer[
        ["parent_text", "text", "customer_tweet_id"]
    ]
    .rename(columns={"text": "customer_message"})
    .sample(
        min(25, len(customer_replying_to_customer)),
        random_state=42
    )
)

Customer messages with previous context: 32014
Replying to previous AppleSupport message: 23388
Replying to another user's message: 8626

--- Customer replying to AppleSupport ---


,parent_text,customer_message,customer_tweet_id
313887,@201222 We're happy to hear in your most recen...,@201222 You're welcome!,359816.0
484492,@180258 Let's look in to this together. Check ...,"@180258 If you're in the US, those features wi...",548174.0
922924,@362414 We'd like to get a closer look. Are yo...,@362414 Send us a DM with the country that yo...,1024356.0
1024067,"@387126 Battery life is important, and we’ll h...",@387126 Go ahead and send us a DM letting us k...,1133667.0
479660,@246093 We're happy to assist. Which iOS versi...,@246093 Absolutely. That's what we're here for!,543051.0
1268538,@445305 We can help with your device. When do ...,@445305 Let us know in DM if iOS 11.0.3 shows ...,1398922.0
1037601,@390357 We'd love to look into this with you. ...,@390357 Great details! We'd like to continue p...,1148847.0
250231,@184690 Let’s take a look at that together. W...,@184690 Thanks. So we can look at all of your ...,288366.0
572188,@272890 Good question. Manage your cellular da...,@272890 What happens when you enable 4G?,644105.0
1224656,@434930 We've got you covered. Please check ou...,@434930 The videos helps a lot. We'd like to c...,1350601.0



--- Customer replying to another user ---


,parent_text,customer_message,customer_tweet_id
2158341,* Me hate how attached me am to my boo me swea...,"@672417 We recently released an iOS update, 11...",2320088.0
1582124,NaN,@118098 Here’s what you can do to work around ...,1734414.0
425385,"@AppleSupport screwed up with iOS11, so many p...",@230320 We want to make sure your iPhone is wo...,484588.0
2585669,Is my phone the only one turning it into I.T?,@771550 We'd like to work with you and look in...,2756831.0
1037607,@AppleSupport how do I deauthorize computers I...,@390360 Hey there! We just wanted to know if y...,1148856.0
1644659,Every device in my life is from @115858 and I ...,"@540176 Hey, we wan to take a look into what's...",1797737.0
1478226,@497394 @497395 @115858 Oh! That’s not bad. Bu...,"@497392 Please send us a DM, we'd like to look...",1623849.0
2158327,Same here ! @AppleSupport what's up. iPhone 7 ...,@672411 Thanks for reaching out. We recently r...,1429801.0
1923003,@AppleSupport I’m having the same trouble that...,@465561 We want your Apple Music working for y...,2077311.0
2137236,@AppleSupport my iPhone 6+ rear camera can’t f...,@593746 Feel free to send us a DM and we can s...,2298576.0


In [31]:
# Build the cleaned AppleSupport interaction dataset
# directly from the existing full dataframe.

# Lookup tables from the already-loaded dataset
tweet_text = df.set_index("tweet_id")["text"]

# Get AppleSupport replies to customer tweets
clean_apple = df[
    (df["author_id"] == "AppleSupport") &
    (df["inbound"] == False) &
    (df["in_response_to_tweet_id"].notna())
].copy()

# The tweet AppleSupport is replying to is the customer's message
clean_apple["customer_tweet_id"] = (
    clean_apple["in_response_to_tweet_id"]
)

clean_apple["customer_message"] = (
    clean_apple["customer_tweet_id"]
    .map(tweet_text)
)

# Apple's actual response
clean_apple["brand_response"] = clean_apple["text"]

# Find whether the customer's message itself was a reply
parent_lookup = (
    df.set_index("tweet_id")["in_response_to_tweet_id"]
)

clean_apple["parent_tweet_id"] = (
    clean_apple["customer_tweet_id"]
    .map(parent_lookup)
)

# Previous tweet text, if available
clean_apple["previous_message"] = (
    clean_apple["parent_tweet_id"]
    .map(tweet_text)
    .fillna("")
    .astype(str)
    .str.strip()
)

# Who wrote the previous tweet?
author_lookup = df.set_index("tweet_id")["author_id"]

clean_apple["parent_author"] = (
    clean_apple["parent_tweet_id"]
    .map(author_lookup)
)

# Identify obvious low-information messages
clean_apple["is_low_info"] = (
    clean_apple["customer_message"]
    .fillna("")
    .str.strip()
    .apply(lambda x: bool(low_info_pattern.match(x)))
)

before_count = len(clean_apple)

# Remove low-information messages
clean_apple = clean_apple[
    ~clean_apple["is_low_info"]
].copy()

# Remove missing/empty interactions
clean_apple = clean_apple[
    clean_apple["customer_message"].notna() &
    clean_apple["customer_message"].str.strip().ne("") &
    clean_apple["brand_response"].notna() &
    clean_apple["brand_response"].str.strip().ne("")
].copy()

# Remove exact duplicate interactions
clean_apple = clean_apple.drop_duplicates(
    subset=[
        "customer_tweet_id",
        "customer_message",
        "brand_response"
    ]
).copy()

# Whether previous conversational context exists
clean_apple["has_context"] = (
    clean_apple["previous_message"].ne("")
)

# Keep only the columns we need
clean_apple = clean_apple[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "has_context",
        "parent_author",
        "brand_response"
    ]
].reset_index(drop=True)

print("Original interactions:", before_count)
print("Clean interactions:", len(clean_apple))
print("Removed:", before_count - len(clean_apple))

print("\nContext availability:")
print(clean_apple["has_context"].value_counts())

print("\nFinal columns:")
print(clean_apple.columns.tolist())

print("\nSample cleaned interactions:")
display(
    clean_apple.sample(
        20,
        random_state=42
    )
)

Original interactions: 106719
Clean interactions: 106523
Removed: 196

Context availability:
has_context
False    74848
True     31675
Name: count, dtype: int64

Final columns:
['customer_tweet_id', 'customer_message', 'previous_message', 'has_context', 'parent_author', 'brand_response']

Sample cleaned interactions:


,customer_tweet_id,customer_message,previous_message,has_context,parent_author,brand_response
43891,1533007.0,@115858 what’s up with this nonstop “I” issue?...,,False,NaN,@475474 Let's look into this together. Reach o...
34665,1224310.0,@AppleSupport my iPhone 7 screen is all white ...,,False,NaN,@407226 Hi there! Let's try out these steps: h...
68276,2000766.0,@461407 @115858 I️ mean write the letter sorry...,@461407 @115858 Mine keeps doing that too! Its...,True,591615,@591615 We can check it out. Meet us in DM wit...
59510,1808936.0,@AppleSupport Hi. There is a speaker grille on...,,False,NaN,@326564 Running into audio issues? Let's get t...
18926,675393.0,Is there some kind of a bug causing random blu...,,False,NaN,@218074 We'd like to help. DM us what iOS vers...
13655,513644.0,@AppleSupport sucks! Getting though to a store...,,False,NaN,@238226 That was not the experience we wanted ...
98665,2711876.0,@AppleSupport Several apps my phone already bl...,@761386 Can you tell us if it is happening on ...,True,AppleSupport,@761386 Please DM us so we can gather further ...
88255,2404846.0,My phone is so slow and glitchy now because of...,,False,NaN,@691713 We're happy to assist. Which specific ...
47797,1617118.0,@115858 ya’ll need to get it together 🙄 I’m si...,,False,NaN,@495875 Thanks for reaching out to us. We have...
36542,1301894.0,@AppleSupport @115858 left AirPod trouble -&gt...,,False,NaN,@424256 Which device do you use with your AirP...


In [32]:
from pathlib import Path

# Create processed-data directory if it doesn't exist
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save the cleaned AppleSupport interactions
output_path = processed_dir / "apple_support_clean.csv"

clean_apple.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")
print(f"Rows: {len(clean_apple):,}")
print(f"Columns: {len(clean_apple.columns)}")

Saved: ..\data\processed\apple_support_clean.csv
Rows: 106,523
Columns: 6
